Thông tin về bộ dữ liệu:

Nơi lưu: /content/test-2/test

Định dạng: YOYOv11

Bài toán: Xác định nhiều loại đối tượng

Số nhãn: 7

Danh sách nhãn (id từ 0 -> 6)

['Checkbox', 'Image', 'Input-Field', 'Radio-Button', 'Select-Dropdown', 'Text', 'labelled-buttons']

In [ ]:
import os
from google.colab import userdata

os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("detect-rhm8j").project("test-bv1nt")
version = project.version(2)
dataset = version.download("yolov11")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 91.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 131.0 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to test-2 in yolov11:: 100%|██████████| 98/98 [00:00<00:00, 7547.17it/s]


In [ ]:
!pip install easyOCR
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 963.8/963.8 kB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.1/292.1 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 32.2 MB/s eta 0:00:00


In [ ]:
import cv2
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# TH1: Chỉ dùng EasyOCR

In [ ]:
import easyocr
import time
import os

# Initialize EasyOCR reader with GPU enabled
reader = easyocr.Reader(['en'], gpu=True) # Set gpu=True to use GPU


In [21]:
%%time

dataset_path = f"{dataset.location}/test/images"
image_files = [os.path.join(dataset_path, f) for f in os.listdir(dataset_path) if f.endswith(('.jpg', '.jpeg', '.png'))]

print(f"Starting OCR on {len(image_files)} images...")

# Record start time
start_time = time.time()

num_objects = 0

for img_path in image_files:
    # Read text from the image
    result = reader.readtext(img_path)

    num_objects += len(result)

print(f"Total number of objects detected: {num_objects}")

# Record end time
end_time = time.time()

print("\nOCR complete.")

# Calculate and display overall processing time
total_processing_time = end_time - start_time
print(f"Total OCR Processing Time for all images: {total_processing_time:.4f} seconds")

Starting OCR on 46 images...
Total number of objects detected: 1520

OCR complete.
Total OCR Processing Time for all images: 19.1955 seconds
CPU times: user 19.1 s, sys: 40.4 ms, total: 19.1 s
Wall time: 19.2 s


In [22]:
print(f"Total number of objects detected: {num_objects}")

Total number of objects detected: 1520


#TH2: Thực hiện xác định đối tượng trước khi OCR

In [ ]:
from ultralytics import YOLO

# Load a model
model = YOLO('yolo.pt')  # load a pretrained model (recommended for training)

# Use the model
model.info()

YOLO11n summary: 181 layers, 2,591,205 parameters, 0 gradients, 6.4 GFLOPs


(181, 2591205, 0, 6.4470528)

In [ ]:
import os
import time
from PIL import Image
import numpy as np
import cv2

In [23]:
%%time

dataset_path = f"{dataset.location}/test/images"
image_files = [os.path.join(dataset_path, f) for f in os.listdir(dataset_path) if f.endswith(('.jpg', '.jpeg', '.png'))]

print(f"Starting predictions and selective OCR on {len(image_files)} images...")

# Record start time for prediction and OCR
start_time = time.time()

# Define the classes for OCR
classes_for_ocr = [5, 6] # 5 for Text, 6 for labelled-buttons

total_ocr_time = 0
text_object_count = 0
labelled_buttons_count = 0

for img_path in image_files:
    image = Image.open(img_path)
    # Perform prediction
    detections = model.predict(image)

    # Process detections and perform OCR on specified classes
    for detection in detections:
        if detection.boxes is not None:
            for box in detection.boxes:
                class_id = int(box.cls)
                if class_id in classes_for_ocr:
                    # Get bounding box coordinates
                    x1, y1, x2, y2 = [int(i) for i in box.xyxy[0]]
                    # Crop the detected region
                    cropped_img = image.crop((x1, y1, x2, y2))
                    # Convert PIL Image to numpy array
                    cropped_img_np = cv2.cvtColor(np.array(cropped_img), cv2.COLOR_RGB2BGR)
                    # Perform OCR on the cropped region
                    ocr_start_time = time.time()
                    result = reader.readtext(cropped_img_np)
                    # Record end time for individual OCR
                    ocr_end_time = time.time()
                    total_ocr_time += (ocr_end_time - ocr_start_time)

                    if class_id == 5:
                        text_object_count += 1
                    elif class_id == 6:
                        labelled_buttons_count += 1


# Record end time for prediction and OCR
end_time = time.time()

print("\nPrediction and selective OCR complete.")

# Calculate and display overall processing time
total_processing_time = end_time - start_time
print(f"Total Processing Time for all images (Prediction + Selective OCR): {total_processing_time:.4f} seconds")
print(f"Total OCR Time (Selective OCR only): {total_ocr_time:.4f} seconds")
print(f"Total number of Text objects detected: {text_object_count}")
print(f"Total number of Labelled Buttons objects detected: {labelled_buttons_count}")

Starting predictions and selective OCR on 46 images...

0: 640x640 76 Checkboxs, 3 Input-Fields, 83 Texts, 8 labelled-buttonss, 10.7ms
Speed: 2.2ms preprocess, 10.7ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 3 Input-Fields, 22 Texts, 7 labelled-buttonss, 7.8ms
Speed: 1.4ms preprocess, 7.8ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 14 Checkboxs, 8 Input-Fields, 41 Texts, 3 labelled-buttonss, 8.0ms
Speed: 1.3ms preprocess, 8.0ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 10 Checkboxs, 7 Input-Fields, 6 Select-Dropdowns, 36 Texts, 12 labelled-buttonss, 14.7ms
Speed: 1.9ms preprocess, 14.7ms inference, 1.9ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 9 Checkboxs, 6 Input-Fields, 5 Select-Dropdowns, 33 Texts, 10 labelled-buttonss, 10.8ms
Speed: 1.9ms preprocess, 10.8ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 640)

0: 640x640 2 Checkboxs, 18 Texts, 1 